<a href="https://colab.research.google.com/github/92-vasim/LLMs/blob/main/Generative-AI-With-LLM-DLAI/Lab_1_summarize_dialogue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %pip install --upgrade pip
# %pip install torch==1.13.1 torchdata==8.5.1 --quiet

# %pip install transformers==4.27.2 datasets==2.11.0

In [ ]:
# %pip install datasets==2.11.0

In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSeq2SeqLM
from transformers import AutoTokenizer
from transformers import GenerationConfig

In [ ]:
dataset_name = "knkarthick/dialogsum"

dataset = load_dataset(dataset_name)

In [ ]:
example_indices = [40, 200]

dash_line = "-".join('' for x in range(100))

for i, idx in enumerate(example_indices):
    print(dash_line)
    print("Example", i+1)
    print(dash_line)
    print("INPUT DIALOGUE:")
    print(dataset['test'][idx]['dialogue'])
    print("BASELINE HUMAN SUMMARY:")
    print(dataset['test'][idx]['summary'])
    print(dash_line)
    print()

In [ ]:
model_name = "google/flan-t5-base"

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
sentence = "What time is it, Tom?"

sentence_encoded = tokenizer(sentence, return_tensors='pt')

sentence_decoded = tokenizer.decode(
    sentence_encoded['input_ids'][0],
    skip_special_tokens=True
)

print("ENCODED SENTENCE: ")
print(sentence_encoded['input_ids'][0])
print("\nDECODED SENTENCE: ")
print(sentence_decoded)

In [ ]:
for i, idx in enumerate(example_indices):
    dialogue = dataset['test'][idx]['dialogue']
    summary = dataset['test'][idx]['summary']

    inputs = tokenizer(dialogue, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(
            inputs['input_ids'],
            max_new_tokens=50,
        )[0],
        skip_special_tokens=True
    )   

    print(dash_line)
    print("Example", i+1)
    print(dash_line)
    print(f"INPUT PROMPT:\n{dialogue}")
    print(dataset['test'][idx]['dialogue'])
    print(f"BASELINE HUMAN SUMMARY:\n{summary}")
    print(dash_line)
    print(f"MODEL GENERATION - WITHOUT PROMPT ENGINEERING: \n{output}\n")
        

### Summarization with prompt engineering

In [ ]:
for i, idx in enumerate(example_indices):
    dialogue = dataset['test'][idx]['dialogue']
    summary = dataset['test'][idx]['summary']

    prompt = f""" 
Summarize the following conversation.

{dialogue}

Summary:
"""

    inputs = tokenizer(prompt, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(
            inputs['input_ids'],
            max_new_tokens=50,
        )[0],
        skip_special_tokens=True
    )   

    print(dash_line)
    print("Example", i+1)
    print(dash_line)
    print(f"INPUT PROMPT:\n{prompt}")
    print(dataset['test'][idx]['dialogue'])
    print(f"BASELINE HUMAN SUMMARY:\n{summary}")
    print(dash_line)
    print(f"\nMODEL GENERATION - ZERO SHOT: \n{output}\n")
        

#### Different prompt

In [ ]:
for i, idx in enumerate(example_indices):
    dialogue = dataset['test'][idx]['dialogue']
    summary = dataset['test'][idx]['summary']

    prompt = f""" 
Dialogue: 

{dialogue}

What was going on?
"""

    inputs = tokenizer(prompt, return_tensors='pt')
    output = tokenizer.decode(
        model.generate(
            inputs['input_ids'],
            max_new_tokens=50,
        )[0],
        skip_special_tokens=True
    )   

    print(dash_line)
    print("Example", i+1)
    print(dash_line)
    print(f"INPUT PROMPT:\n{prompt}")
    print(dataset['test'][idx]['dialogue'])
    print(f"BASELINE HUMAN SUMMARY:\n{summary}")
    print(dash_line)
    print(f"\nMODEL GENERATION - ZERO SHOT: \n{output}\n")
        

#### ONE SHOT

In [ ]:
def make_prompt(example_indices_full, example_idx_to_summarize):
    prompt = ""
    for i, idx in enumerate(example_indices_full):
        dialogue = dataset['test'][idx]['dialogue']
        summary = dataset['test'][idx]['summary']

        """ 
        The stop sequence '{summary}\n\n\n' is important for FLAN-T5. Other models may have their own preferred stop sequence.
        """
        prompt += f"""
Dialogue: 

{dialogue}

What was going on?
{summary}


"""
        
    dialogue = dataset['test'][example_idx_to_summarize]['dialogue']
    prompt += f"""
Dialogue: 

{dialogue}

What was going on? 
"""
    return prompt 

In [ ]:
example_indices_full = [40]
example_idx_to_summarize = 200

one_shot_prompt = make_prompt(example_indices_full, example_idx_to_summarize)

print(one_shot_prompt)

In [ ]:
summary = dataset['test'][example_idx_to_summarize]['summary']

inputs = tokenizer(one_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(
        inputs['input_ids'],
        max_new_tokens=50,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{summary}")
print(dash_line)
print(f"\nMODEL GENERATION - ONE SHOT: \n{output}\n")

#### FEW SHOT

In [ ]:
example_indices_full = [40, 80, 120]
example_idx_to_summarize = 200

few_shot_prompt = make_prompt(example_indices_full, example_idx_to_summarize)

print(few_shot_prompt)

In [ ]:
summary = dataset['test'][example_idx_to_summarize]['summary']

inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(
        inputs['input_ids'],
        max_new_tokens=50,
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{summary}")
print(dash_line)
print(f"\nMODEL GENERATION - FEW SHOT: \n{output}\n")

### Generative Configuration Parameters for Inference

In [ ]:
generation_config = GenerationConfig(max_new_tokens=50)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.1)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=0.5)
# generation_config = GenerationConfig(max_new_tokens=50, do_sample=True, temperature=1.0)

# optional
# model.generation_config.decoder_start_token_id = tokenizer.decoder_start_token_id

inputs = tokenizer(few_shot_prompt, return_tensors='pt')
output = tokenizer.decode(
    model.generate(
        inputs['input_ids'],
        generation_config=generation_config
    )[0],
    skip_special_tokens=True
)

print(dash_line)
print(f"BASELINE HUMAN SUMMARY:\n{summary}")
print(dash_line)
print(f"\nMODEL GENERATION - FEW SHOT: \n{output}\n")